# Decompression relay with heterogeneous solid

Notebook for the decompression relay function. Last updated 22/10/2024.

Start with dependencies:

In [1]:
import numpy as np
import pyDEW
import os
import matplotlib.pyplot as plt
from thermoengine import model
from copy import deepcopy
import pandas as pd
import linecache
from scipy.interpolate import LinearNDInterpolator
from timeit import default_timer as timer
import warnings

# Additional functions for pyDEW
import pyDEW_functions as pDF

# Structural functions for the relay
import relay_functions as rF

# Folder to export results
if not os.path.exists("./data"):
    os.makedirs("./data")

# pyDEW flags up a few things when the system is created - this counteracts it
warnings.simplefilter(action='ignore', category=FutureWarning)
pd.options.mode.chained_assignment = None

Create our pyDEW system (required for DATA0 and DATA1 files).

In [2]:
# Adjust here the basis species in the DEW system - deepcopy to avoid mucking up defaults
defaultBasisSpecies = pyDEW.defaultsystem.basis_species_names
basisSpecies = deepcopy(defaultBasisSpecies)

defaultOtherSpecies = pyDEW.defaultsystem.other_species_names
otherSpecies = deepcopy(defaultOtherSpecies)

# Add missing Mg complex to the other species
otherSpecies.append('MG(OH)2(AQ)')

# Switch basis species - replace CO2(AQ) with CO3-2 directly to avoid pyDEW issues
basisSpecies[basisSpecies.index('CO2(AQ)')] = 'CO3-2'

# Create DEW system
dew_system = pyDEW.System(basis_species=basisSpecies, other_species=otherSpecies)


Create the reactants that will fill our column.

In [3]:
# List of reactants formed from mineral dictionaries
olvDict = {'reactant': 'OLIVINE(SS)',
           'moles': 5.9,
           'composition': {'FORSTERITE': 0.9, 'FAYALITE': 0.1}}

opxDict = {'reactant': 'ORTHOPYROXENE(SS)',
           'moles': 1.9,
           'composition': {'ENSTATITE_OR': 0.9, 'FERROSILITE': 0.1}}

# cpxDict = {'reactant': 'CLINOPYROXENE(SS)',
#            'moles': 0.2,
#            'composition': {'DIOPSIDE': 0.9, 'HEDENBERGITE': 0.1}}

# spnDict = {'reactant': 'SPINEL',
#            'moles': 0.04}

mgtDict = {'reactant': 'MAGNETITE',
           'moles': 0.05}

atgDict = {'reactant': 'ANTIGORITE',
           'moles': 0.95}


harzReactant = [olvDict, opxDict]#, cpxDict, spnDict] #, mgtDict] # Basically the lherzolite minus spinel and cpx
serpReactant = [atgDict, mgtDict] # made up of just antigorite and magnetite

In [4]:
pressures = [20000.0]*100
temperatures = [450.0+273.15]*100
fluidComposition = rF.createHybridFluid([0,0,1], pressures[0], temperatures[0])
# This column is 50 steps harzburgite and 50 steps serpentinite
startingComposition = rF.createHeterogeneousInput([harzReactant, serpReactant],
                                                  dew_system,
                                                  [1000.0, 1000.0],
                                                  [4, 1],
                                                  pressures,
                                                  temperatures)

In [5]:
rF.decompressionRelay(startingComposition, fluidComposition[0], fluidComposition[1],
                      dew_system, timesteps=3, solventMass=1000, terminalFRR=0.2,
                      exportPath='test')

-----RELAY STARTING-----
Relay function initiated
Starting from scratch...
t=1, calculations=18, errors=82 of which FRR passes=82
t=2, calculations=34, errors=66 of which FRR passes=66
No mineral product found.
No mineral product found.
No mineral product found.
No mineral product found.
No mineral product found.
No mineral product found.
No mineral product found.
t=3, calculations=43, errors=57 of which FRR passes=50
-----RELAY COMPLETE-----
Total iterations : 300
Total successful DEW-EQ3/6 calculations : 95
Total runtime : 224.6 seconds
Total runtime : 3.7 minutes
Data exported to: data/test.xlsx

